### Imports

In [43]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo






from dotenv import load_dotenv
import os,asyncio


### Loading Language Model


In [44]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

# Create a client with your local Ollama model
ollama_client = OllamaChatCompletionClient(
    model="llama3.1:latest"
)
 

In [45]:

load_dotenv()
model_client = OpenAIChatCompletionClient(
    model="gemini-2.5-flash",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",  # Add this
    api_key=os.environ['GOOGLE_API_KEY'],
    model_info=ModelInfo(vision=True, function_calling=True, json_output=True, family="gemini-2.5-flash", structured_output=True)
)


### Agents

In [ ]:
from tools import get_additive_manufacturing_equipment,get_timetable,add_booking
import pandas as pd

 
# machineDecider = AssistantAgent(
#     name="weather_agent",
#     model_client=ollama_client,
#     tools=[get_additive_manufacturing_equipment],
#     system_message = """You are a helpful assistant who decides 
#     what kind of tools will be required to manufacture something.""",
#     reflect_on_tool_use=True,
#     model_client_stream=True,  # Enable streaming tokens from the model client.
# )


system_message = """
You are a helpful assistant who analyses the timetable and adds bookings. 
Step 1: Call get_timetable to retrieve the schedule.
Step 2: Determine the earliest 3-hour free slot after 16:00.
Step 3: Always return a structured FunctionCall for add_booking to book the slot.
- Use arguments: name, start_time, duration_hours
- Do NOT describe the booking in text; the only output for booking must be a FunctionCall
- If add_booking returns a string, tell the user if it is 'success'
"""




time_finder = AssistantAgent(
    name="time_agent",
    model_client=model_client,
    tools=[get_timetable,add_booking],
    system_message = system_message,
    reflect_on_tool_use=True,
    model_client_stream=True,  # Enable streaming tokens from the model client.
)


























### Running system

In [ ]:


# mas = MultiAgentSystem(agents=[agent_a, agent_b])

# # Agent A sends a message to Agent B
# mas.send_message(from_agent="AgentA", to_agent="AgentB", message="Please summarize sales data.")



#task="Create a model of a figurine and resin print it"

# Run the agent and stream the messages to the console.
async def main() -> None:
    await Console(time_finder.run_stream(task=f"Check the timetable using the proper funcition. then Book a 3 hour slor for marley when the starting time is 4pm at the earliest free day using the add booking fucnito"))
    # Close the connection to the model client.
    await model_client.close()


#asyncio.run(main())
await main()  # DO NOT use asyncio.run(main())

---------- TextMessage (user) ----------
Check the timetable using the proper funcition. then Book a 3 hour slor for marley when the starting time is 4pm at the earliest free day using the add booking fucnito


---------- ToolCallRequestEvent (time_agent) ----------
[FunctionCall(id='function-call-10801864465039705863', arguments='{}', name='get_timetable')]
---------- ToolCallExecutionEvent (time_agent) ----------
[FunctionExecutionResult(content='        Time Monday Tuesday Wednesday  Thursday Friday Saturday  Sunday\n0   00:00:00    NaN     NaN     Hello       NaN    NaN      NaN     NaN\n1   00:30:00    NaN     NaN     Hello       NaN    NaN    Hello     NaN\n2   01:00:00    NaN     NaN     Hello       NaN    NaN    Hello     NaN\n3   01:30:00    NaN     NaN     Hello       NaN    NaN    Hello     NaN\n4   02:00:00    NaN     NaN       NaN       NaN    BOB    Hello     NaN\n5   02:30:00    NaN     NaN       NaN       NaN    BOB      NaN     NaN\n6   03:00:00    NaN     NaN       NaN       NaN    BOB      NaN     NaN\n7   03:30:00    BOB     NaN     Hello       NaN    BOB      NaN     NaN\n8   04:00:00    NaN     NaN     Hello       NaN    BOB      NaN     NaN\n9   04:30:00    NaN     NaN 

C:\Users\admin\AppData\Local\Temp\ipykernel_5108\3423679471.py:12: UserWarning: No text content or tool calls are available. Model returned empty result.
  await Console(time_finder.run_stream(task=f"Check the timetable using the proper funcition. then Book a 3 hour slor for marley when the starting time is 4pm at the earliest free day using the add booking fucnito"))


In [48]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

def fill_and_style(file_path, sheet_name, day, start_time, end_time, name ):

    wb = load_workbook(file_path)
    ws = wb[sheet_name]


    week_map = {
   
    "monday": "B",
    "tuesday": "C",
    "wednesday": "D",
    "thursday": "E",
    "friday": "F",
    "saturday": "G",
    "sunday": "H",
    }


    column=week_map[day]

    def time_string_to_row(t):
        hour, minute = map(int, t.split(":"))
        return (hour * 2 + minute // 30) + 2

    start_row=time_string_to_row(start_time)
    end_row=time_string_to_row(end_time)

    for row in range(start_row, end_row + 1):
        cell = ws[f"{column}{row}"]
        cell.value = name

    wb.save(file_path)

fill_and_style("test.xlsx", "Sheet1", "monday", "10:00", "11:00", "Alice")


In [49]:
import pandas as pd

df = pd.read_excel("test.xlsx", sheet_name="Sheet1")
print(type(df))

<class 'pandas.DataFrame'>
